In [26]:
from dotenv import load_dotenv
import os
import requests
from pprint import pprint
import pandas as pd
import missingno as mno
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import time

load_dotenv()

True

In [2]:
API_KEY = os.getenv("TICKETMASTER_API_KEY")

if not API_KEY:
    raise ValueError("No API key found — check your .env file exists and has TICKETMASTER_API_KEY set")

In [3]:
response = requests.get(
    "https://app.ticketmaster.com/discovery/v2/events.json",
    params={
        "apikey": API_KEY,
        "countryCode": "GB",
        "classificationName": "Music",
        "size": 20,
        "page": 0
    }
)
print(response.status_code)
data = response.json()

200


In [10]:
# Exploring the structure of the data, and what data structures are embedded

data.keys()
data = response.json()

print(type(data))        # <class 'dict'>
print(data.keys())       # dict_keys(['_embedded', '_links', 'page'])

print(type(data['_embedded']))       # <class 'dict'>
print(data['_embedded'].keys())      # dict_keys(['events'])

print(type(data['_embedded']['events']))     # <class 'list'>
print(len(data['_embedded']['events']))      # e.g. 20

print(type(data['_embedded']['events'][0]))  # <class 'dict'>  <- one single event
print(data['_embedded']['events'][0].keys())

<class 'dict'>
dict_keys(['_embedded', '_links', 'page'])
<class 'dict'>
dict_keys(['events'])
<class 'list'>
20
<class 'dict'>
dict_keys(['name', 'type', 'id', 'test', 'url', 'locale', 'images', 'sales', 'dates', 'classifications', 'promoter', 'promoters', 'pleaseNote', 'products', 'accessibility', 'ticketLimit', 'ageRestrictions', 'ticketing', 'nameOrigin', 'ticketTextLines', '_links', '_embedded'])


In [11]:
pprint(data)

{'_embedded': {'events': [{'_embedded': {'attractions': [{'_links': {'self': {'href': '/discovery/v2/attractions/K8vZ9171Qi7?locale=en-us'}},
                                                          'classifications': [{'family': False,
                                                                               'genre': {'id': 'KnvZfZ7vAv1',
                                                                                         'name': 'Hip-Hop/Rap'},
                                                                               'primary': True,
                                                                               'segment': {'id': 'KZFzniwnSyZfZ7v7nJ',
                                                                                           'name': 'Music'},
                                                                               'subGenre': {'id': 'KZazBEonSMnZfZ7vkdA',
                                                                                            'n

In [12]:
# Creating the dataframe df

events = data['_embedded']['events']  # your list of 20 dicts

df = pd.json_normalize(events)
df.head()

,name,type,id,test,url,locale,images,classifications,promoters,pleaseNote,...,seatmap.staticUrl,accessibility.ticketLimit,linkMoreInfo.descriptions.ca-es,linkMoreInfo.descriptions.de-de,linkMoreInfo.descriptions.en-es,linkMoreInfo.descriptions.en-us,linkMoreInfo.descriptions.en-de,linkMoreInfo.descriptions.es-es,linkMoreInfo.descriptions.en-gb,linkMoreInfo.url
0,JAY-Z - 30,event,17u8v0G6CksBAD4,False,https://www.ticketmaster.co.uk/jayz-30-london-...,en-us,"[{'ratio': '16_9', 'url': 'https://s1.ticketm....","[{'primary': True, 'segment': {'id': 'KZFzniwn...","[{'id': '6678', 'name': 'LIVE NATION UK FOR AR...",There will be no admittance for children under...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,The World of Hans Zimmer - a New Dimension,event,G5vHZbJBpZsXN,False,https://www.ticketmaster.co.uk/the-world-of-ha...,en-us,"[{'ratio': '16_9', 'url': 'https://s1.ticketm....","[{'primary': True, 'segment': {'id': 'KZFzniwn...","[{'id': '6145', 'name': 'KMJ ENTERTAINMENT LTD...",Under 14s must be accompanied by an adult over...,...,https://s1.ticketm.net/uk/tmimages/venue/maps/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,The World of Hans Zimmer - a New Dimension,event,1AeGZbJGkB_ZZUA,False,https://www.ticketmaster.co.uk/the-world-of-ha...,en-us,"[{'ratio': '16_9', 'url': 'https://s1.ticketm....","[{'primary': True, 'segment': {'id': 'KZFzniwn...","[{'id': '6145', 'name': 'KMJ ENTERTAINMENT LTD...",Under 14s must be accompanied by an adult over...,...,https://s1.ticketm.net/uk/tmimages/venue/maps/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,The Holiday in Concert - Film with Live Orchestra,event,G5vHZ_aedKoux,False,https://www.ticketmaster.co.uk/the-holiday-in-...,en-us,"[{'ratio': '16_9', 'url': 'https://s1.ticketm....","[{'primary': True, 'segment': {'id': 'KZFzniwn...","[{'id': '3013', 'name': 'SENBLA LIMITED', 'des...",Under 16s must be accompanied by an adult over...,...,https://s1.ticketm.net/uk/tmimages/venue/maps/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,The Holiday - Film With Live Orchestra,event,1AdbZ_kGkRuAbPW,False,https://www.ticketmaster.co.uk/the-holiday-fil...,en-us,"[{'ratio': '16_9', 'url': 'https://s1.ticketm....","[{'primary': True, 'segment': {'id': 'KZFzniwn...","[{'id': '3013', 'name': 'SENBLA LIMITED', 'des...",Over 12s only. Under 14s must be accompanied b...,...,https://s1.ticketm.net/uk/tmimages/venue/maps/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 59 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   name                                   20 non-null     object 
 1   type                                   20 non-null     object 
 2   id                                     20 non-null     object 
 3   test                                   20 non-null     bool   
 4   url                                    20 non-null     object 
 5   locale                                 20 non-null     object 
 6   images                                 20 non-null     object 
 7   classifications                        20 non-null     object 
 8   promoters                              20 non-null     object 
 9   pleaseNote                             20 non-null     object 
 10  products                               1 non-null      object 
 11  nameOrig

Pagination required to pull data from all the pages

In [17]:
def generate_month_chunks(year, months):
    chunks = []
    for month in months:
        start = pd.Timestamp(year=year, month=month, day=1)
        end = (start + pd.offsets.MonthEnd(1)).replace(hour=23, minute=59, second=59)
        chunks.append((
            start.strftime("%Y-%m-%dT%H:%M:%SZ"),
            end.strftime("%Y-%m-%dT%H:%M:%SZ")
        ))
    return chunks

date_chunks = generate_month_chunks(2026, months=[6, 7, 8])  # June, July, August
print(date_chunks)

[('2026-06-01T00:00:00Z', '2026-06-30T23:59:59Z'), ('2026-07-01T00:00:00Z', '2026-07-31T23:59:59Z'), ('2026-08-01T00:00:00Z', '2026-08-31T23:59:59Z')]


In [20]:
def fetch_page(page_number, start_date, end_date):
    params = {
        "apikey": API_KEY,
        "countryCode": "GB",
        "classificationName": "Music",
        "startDateTime": start_date,
        "endDateTime": end_date,
        "sort": "date,asc",
        "size": 200,
        "page": page_number
    }
    response = requests.get(BASE_URL, params=params)

    if response.status_code == 429:
        print("Rate limited — waiting 5s...")
        time.sleep(5)
        return fetch_page(page_number, start_date, end_date)

    response.raise_for_status()
    return response.json()

In [21]:
def fetch_all_pages_for_range(start_date, end_date):
    all_pages = []
    first_page = fetch_page(0, start_date, end_date)
    all_pages.append(first_page)

    total_pages = first_page.get("page", {}).get("totalPages", 1)
    total_elements = first_page.get("page", {}).get("totalElements", 0)
    print(f"  {start_date[:7]}: {total_elements} events across {total_pages} pages")

    max_pages = total_pages  # no cap — pull everything for a 3-month window

    for page_num in range(1, max_pages):
        time.sleep(0.25)
        page_data = fetch_page(page_num, start_date, end_date)
        all_pages.append(page_data)

    return all_pages

In [27]:
all_events = []
all_pages_flat = []
BASE_URL = "https://app.ticketmaster.com/discovery/v2/events.json"

for start_date, end_date in date_chunks:
    pages = fetch_all_pages_for_range(start_date, end_date)
    all_pages_flat.extend(pages)
    for page in pages:
        all_events.extend(page.get("_embedded", {}).get("events", []))
    time.sleep(0.5)

print(f"\nTotal events collected: {len(all_events)}")

  2026-06: 1 events across 1 pages
  2026-07: 988 events across 5 pages
  2026-08: 1101 events across 6 pages

Total events collected: 2090


In [28]:
RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
# now save raw
for i, page in enumerate(all_pages_flat):
    with open(RAW_DIR / f"uk_music_jun_aug_page{i}.json", "w") as f:
        json.dump(page, f)

In [29]:
from collections import Counter

month_counts = Counter()
for event in all_events:
    date = event.get("dates", {}).get("start", {}).get("localDate")
    if date:
        month_counts[date[:7]] += 1

for month in sorted(month_counts):
    print(month, month_counts[month])

2026-06 2
2026-07 995
2026-08 1093


In [30]:
event_ids = [e.get('id') for e in all_events]
print(f"Total events: {len(event_ids)}")
print(f"Unique event IDs: {len(set(event_ids))}")

Total events: 2090
Unique event IDs: 2081


In [31]:
# confirming the pagination worked
response = requests.get(
    "https://app.ticketmaster.com/discovery/v2/events.json",
    params={
        "apikey": API_KEY,
        "countryCode": "GB",
        "classificationName": "Music",
        "startDateTime": "2026-07-01T00:00:00Z",
        "endDateTime": "2026-08-31T23:59:59Z",
        "size": 200,
        "page": 0
    }
)

data = response.json()
page_info = data.get("page", {})

print(f"Total elements matched: {page_info.get('totalElements')}")
print(f"Total pages (at size=200): {page_info.get('totalPages')}")

Total elements matched: 2081
Total pages (at size=200): 11


In [32]:
# 1. Does the count match what totalElements said we should expect?
print(f"Expected total (from API metadata): 2081")
print(f"Actually collected in all_events: {len(all_events)}")

# 2. Did we genuinely make 11 separate page requests, not just 1?
print(f"Number of raw pages fetched: {len(all_pages_flat)}")

# 3. Check each page's own reported page number, to confirm they're actually different pages, not the same one repeated
for page in all_pages_flat:
    page_num = page.get("page", {}).get("number")
    events_in_page = len(page.get("_embedded", {}).get("events", []))
    print(f"Page {page_num}: {events_in_page} events")

Expected total (from API metadata): 2081
Actually collected in all_events: 2090
Number of raw pages fetched: 12
Page 0: 1 events
Page 0: 200 events
Page 1: 200 events
Page 2: 200 events
Page 3: 200 events
Page 4: 188 events
Page 0: 200 events
Page 1: 200 events
Page 2: 200 events
Page 3: 200 events
Page 4: 200 events
Page 5: 101 events


Now lets create our df

In [33]:
# Build df from the real paginated pull, not the old 20-event test
df = pd.json_normalize(all_events)
print(df.shape)
df.head()

(2090, 81)


,name,type,id,test,description,url,locale,images,classifications,nameOrigin,...,dates.end.localDate,ageRestrictions.ageRuleDescription,attractionGroups.major.description,attractionGroups.major.descriptions.tm-df,attractionGroups.major.descriptions.en-gb,attractionGroups.minor.id,attractionGroups.minor.name,dates.initialStartDate.localDate,dates.initialStartDate.localTime,dates.initialStartDate.dateTime
0,EVERYWHERE AT ONCE: Dewin,event,LvZ18QxAj1bZeL8vGSGnc,False,Everywhere At Once powered by The National Lot...,https://www.universe.com/events/everywhere-at-...,en-us,"[{'ratio': '16_9', 'url': 'https://s1.ticketm....","[{'segment': {'id': 'KZFzniwnSyZfZ7v7nJ', 'nam...",custom,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,EVERYWHERE AT ONCE: Dewin,event,LvZ18QxAj1bZeL8vGSGnc,False,Everywhere At Once powered by The National Lot...,https://www.universe.com/events/everywhere-at-...,en-us,"[{'ratio': '16_9', 'url': 'https://s1.ticketm....","[{'segment': {'id': 'KZFzniwnSyZfZ7v7nJ', 'nam...",custom,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Orchestral Qawwali Project,event,G5vHZ_dQ2CoCm,False,NaN,https://www.ticketmaster.co.uk/orchestral-qaww...,en-us,"[{'ratio': '16_9', 'url': 'https://s1.ticketm....","[{'primary': True, 'segment': {'id': 'KZFzniwn...",custom,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,The Maccabees,event,G5vHZbM3j8P-r,False,NaN,https://www.ticketmaster.co.uk/the-maccabees-l...,en-us,"[{'ratio': '3_2', 'url': 'https://s1.ticketm.n...","[{'primary': True, 'segment': {'id': 'KZFzniwn...",primaryAttraction,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Billy Ocean with very special guest Marti Pellow,event,G5vHZbVxTtI6b,False,NaN,https://www.ticketmaster.co.uk/billy-ocean-wit...,en-us,"[{'ratio': '3_2', 'url': 'https://s1.ticketm.n...","[{'primary': True, 'segment': {'id': 'KZFzniwn...",custom,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
print(df['name'].nunique()) # We have 1459 unique names - pagination successful

1459


In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2090 entries, 0 to 2089
Data columns (total 81 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   name                                       2090 non-null   object 
 1   type                                       2090 non-null   object 
 2   id                                         2090 non-null   object 
 3   test                                       2090 non-null   bool   
 4   description                                823 non-null    object 
 5   url                                        2090 non-null   object 
 6   locale                                     2090 non-null   object 
 7   images                                     2090 non-null   object 
 8   classifications                            2090 non-null   object 
 9   nameOrigin                                 2090 non-null   object 
 10  sales.public.startDateTi

In [42]:
# null value check
def null_vals(dataframe):
    """
    Show both number of nulls and the percentage of nulls in the whole column across a Pandas dataframe.
    """
    null_vals = dataframe.isnull().sum() # How many nulls in each column
    total_cnt = len(dataframe) # Total entries in the dataframe
    null_vals = pd.DataFrame(null_vals,columns=['null']) # Put the number of nulls in a single dataframe
    null_vals['percent'] = round((null_vals['null']/total_cnt)*100,3) # Round how many nulls are there, as %, of the df
    
    return null_vals.sort_values('percent', ascending=False) # Ordered from MOST to LEAST nulls
    
print(null_vals(df).to_string())

                                           null  percent
dates.initialStartDate.dateTime            2089   99.952
dates.initialStartDate.localTime           2089   99.952
dates.initialStartDate.localDate           2089   99.952
ageRestrictions.ageRuleDescription         2083   99.665
attractionGroups.major.descriptions.en-gb  2063   98.708
attractionGroups.major.descriptions.tm-df  2063   98.708
attractionGroups.major.description         2063   98.708
products                                   2047   97.943
attractionGroups.minor.name                2023   96.794
attractionGroups.minor.id                  2023   96.794
accessibility.url                          1984   94.928
accessibility.urlText                      1984   94.928
dates.end.localDate                        1940   92.823
accessibility.info                         1935   92.584
attractionGroups.attractionId              1919   91.818
attractionGroups.major.id                  1919   91.818
attractionGroups.major.name    

In [44]:
# check to see which columns are nested 

nested_cols = []

for col in df.columns:
    non_null = df[col].dropna()
    if len(non_null) == 0:
        continue
    sample = non_null.iloc[0]
    if isinstance(sample, (list, dict)):
        nested_cols.append(col)

print(nested_cols)

['images', 'classifications', '_links.attractions', '_links.venues', '_embedded.venues', '_embedded.attractions', 'promoters', 'sales.presales', 'products']


In [73]:
# Lets explore the nested columns to see if worth unravelling

df['images'].iloc[0]

[{'ratio': '16_9',
  'url': 'https://s1.ticketm.net/dam/c/797/5e693c26-2881-4776-8f0c-3aa94bfa3797_106511_RETINA_PORTRAIT_16_9.jpg',
  'width': 640,
  'height': 360,
  'fallback': True},
 {'ratio': '4_3',
  'url': 'https://images.universe.com/d3a8bc1e-594f-4671-88c2-fdc233e87e5d/-/format/jpeg/-/scale_crop/305x225/center/-/progressive/yes/-/inline/yes/',
  'width': 305,
  'height': 225,
  'fallback': False},
 {'ratio': '16_9',
  'url': 'https://s1.ticketm.net/dam/c/f50/96fa13be-e395-429b-8558-a51bb9054f50_105951_TABLET_LANDSCAPE_LARGE_16_9.jpg',
  'width': 2048,
  'height': 1152,
  'fallback': True},
 {'ratio': '3_2',
  'url': 'https://images.universe.com/d3a8bc1e-594f-4671-88c2-fdc233e87e5d/-/format/jpeg/-/scale_crop/305x203/center/-/progressive/yes/-/inline/yes/',
  'width': 305,
  'height': 203,
  'fallback': False},
 {'ratio': '16_9',
  'url': 'https://s1.ticketm.net/dam/c/797/5e693c26-2881-4776-8f0c-3aa94bfa3797_106511_RECOMENDATION_16_9.jpg',
  'width': 100,
  'height': 56,
  'fal

Creating seperate tables for nested columns

In [164]:
def build_nested_table(df, id_col, nested_col):
    """
    Turns a nested list-of-dicts (or single dict) column into its own table,
    one row per item, linked back to the parent via id_col.
    Unwraps ONE level of nested dicts (e.g. genre: {id, name} -> genre_id, genre_name).
    Lists nested inside items are skipped (too complex for a flat row).
    """
    rows = []
    for _, row in df.iterrows():
        parent_id = row[id_col]
        value = row.get(nested_col)
        if value is None:
            continue
        items = value if isinstance(value, list) else [value]
        for item in items:
            if not isinstance(item, dict):
                continue
            flat_item = {f"event_{id_col}": parent_id}   # renamed to avoid collision
            for key, val in item.items():
                if isinstance(val, dict):
                    for subkey, subval in val.items():
                        flat_item[f"{key}_{subkey}"] = subval
                elif isinstance(val, list):
                    continue
                else:
                    flat_item[key] = val   # venue's own 'id' now safely separate
            rows.append(flat_item)
    return pd.DataFrame(rows)


In [70]:
df_images = build_nested_table(df, id_col='id', nested_col='images')
df_classifications = build_nested_table(df, id_col='id', nested_col='classifications')
df_venues = build_nested_table(df_og, id_col='id', nested_col='_embedded.venues')
df_attractions = build_nested_table(df, id_col='id', nested_col='_embedded.attractions')
df_promoters = build_nested_table(df, id_col='id', nested_col='promoters')

print("images:", df_images.shape, df_images.columns.tolist())
print("classifications:", df_classifications.shape, df_classifications.columns.tolist())
print("venues:", df_venues.shape, df_venues.columns.tolist())
print("attractions:", df_attractions.shape, df_attractions.columns.tolist())
print("promoters:", df_promoters.shape, df_promoters.columns.tolist())

images: (22683, 7) ['id', 'ratio', 'url', 'width', 'height', 'fallback', 'attribution']
classifications: (2090, 13) ['id', 'segment_id', 'segment_name', 'genre_id', 'genre_name', 'subGenre_id', 'subGenre_name', 'family', 'primary', 'type_id', 'type_name', 'subType_id', 'subType_name']
venues: (2090, 35) ['id', 'name', 'type', 'test', 'url', 'locale', 'postalCode', 'timezone', 'city_name', 'country_name', 'country_countryCode', 'address_line1', 'location_longitude', 'location_latitude', 'upcomingEvents_universe', 'upcomingEvents__total', 'upcomingEvents__filtered', '_links_self', 'address_line2', 'upcomingEvents_ticketmaster', 'ada_adaPhones', 'ada_adaCustomCopy', 'ada_adaHours', 'boxOfficeInfo_openHoursDetail', 'boxOfficeInfo_willCallDetail', 'parkingDetail', 'accessibleSeatingDetail', 'generalInfo_generalRule', 'generalInfo_childRule', 'boxOfficeInfo_acceptedPaymentDetail', 'boxOfficeInfo_phoneNumberDetail', 'social_twitter', 'upcomingEvents_tmr', 'upcomingEvents_tmc', 'upcomingEvents

In [81]:
df_promoters.head()

,id,name,description
0,3251,CUFFE & TAYLOR EVENT MANAGEMENT,CUFFE & TAYLOR EVENT MANAGEMENT / NTL / GBR
1,3251,CUFFE & TAYLOR EVENT MANAGEMENT,CUFFE & TAYLOR EVENT MANAGEMENT / NTL / GBR
2,4132,LLANGOLLEN INTERNATIONAL MUSICAL EISTEDDFOD,LLANGOLLEN INTERNATIONAL MUSICAL EISTEDDFOD / ...
3,5279,S.J.M. LTD,S.J.M. LTD / NTL / GBR
4,5279,S.J.M. LTD,S.J.M. LTD / NTL / GBR


In [80]:
print(null_vals(df_promoters).to_string())

             null  percent
id              0      0.0
name            0      0.0
description     0      0.0


Creating df_og to represent our original df. We will be tweaking df though.

In [82]:
df_new = df.copy()

In [90]:
nested_cols_to_drop = ['images', 'classifications', '_embedded.venues', '_embedded.attractions', 'promoters']

df = df_new.drop(columns=nested_cols_to_drop, errors='ignore')
df_new.shape
df_new.head()

,name,type,id,test,description,url,locale,images,classifications,nameOrigin,...,dates.end.localDate,ageRestrictions.ageRuleDescription,attractionGroups.major.description,attractionGroups.major.descriptions.tm-df,attractionGroups.major.descriptions.en-gb,attractionGroups.minor.id,attractionGroups.minor.name,dates.initialStartDate.localDate,dates.initialStartDate.localTime,dates.initialStartDate.dateTime
0,EVERYWHERE AT ONCE: Dewin,event,LvZ18QxAj1bZeL8vGSGnc,False,Everywhere At Once powered by The National Lot...,https://www.universe.com/events/everywhere-at-...,en-us,"[{'ratio': '16_9', 'url': 'https://s1.ticketm....","[{'segment': {'id': 'KZFzniwnSyZfZ7v7nJ', 'nam...",custom,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,EVERYWHERE AT ONCE: Dewin,event,LvZ18QxAj1bZeL8vGSGnc,False,Everywhere At Once powered by The National Lot...,https://www.universe.com/events/everywhere-at-...,en-us,"[{'ratio': '16_9', 'url': 'https://s1.ticketm....","[{'segment': {'id': 'KZFzniwnSyZfZ7v7nJ', 'nam...",custom,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Orchestral Qawwali Project,event,G5vHZ_dQ2CoCm,False,NaN,https://www.ticketmaster.co.uk/orchestral-qaww...,en-us,"[{'ratio': '16_9', 'url': 'https://s1.ticketm....","[{'primary': True, 'segment': {'id': 'KZFzniwn...",custom,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,The Maccabees,event,G5vHZbM3j8P-r,False,NaN,https://www.ticketmaster.co.uk/the-maccabees-l...,en-us,"[{'ratio': '3_2', 'url': 'https://s1.ticketm.n...","[{'primary': True, 'segment': {'id': 'KZFzniwn...",primaryAttraction,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Billy Ocean with very special guest Marti Pellow,event,G5vHZbVxTtI6b,False,NaN,https://www.ticketmaster.co.uk/billy-ocean-wit...,en-us,"[{'ratio': '3_2', 'url': 'https://s1.ticketm.n...","[{'primary': True, 'segment': {'id': 'KZFzniwn...",custom,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [86]:
df.head()

,name,type,id,test,description,url,locale,nameOrigin,sales.public.startDateTime,sales.public.startTBD,...,dates.end.localDate,ageRestrictions.ageRuleDescription,attractionGroups.major.description,attractionGroups.major.descriptions.tm-df,attractionGroups.major.descriptions.en-gb,attractionGroups.minor.id,attractionGroups.minor.name,dates.initialStartDate.localDate,dates.initialStartDate.localTime,dates.initialStartDate.dateTime
0,EVERYWHERE AT ONCE: Dewin,event,LvZ18QxAj1bZeL8vGSGnc,False,Everywhere At Once powered by The National Lot...,https://www.universe.com/events/everywhere-at-...,en-us,custom,2026-06-04T11:06:18Z,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,EVERYWHERE AT ONCE: Dewin,event,LvZ18QxAj1bZeL8vGSGnc,False,Everywhere At Once powered by The National Lot...,https://www.universe.com/events/everywhere-at-...,en-us,custom,2026-06-04T11:06:18Z,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Orchestral Qawwali Project,event,G5vHZ_dQ2CoCm,False,NaN,https://www.ticketmaster.co.uk/orchestral-qaww...,en-us,custom,2026-01-23T15:00:00Z,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,The Maccabees,event,G5vHZbM3j8P-r,False,NaN,https://www.ticketmaster.co.uk/the-maccabees-l...,en-us,primaryAttraction,2025-11-28T10:00:00Z,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Billy Ocean with very special guest Marti Pellow,event,G5vHZbVxTtI6b,False,NaN,https://www.ticketmaster.co.uk/billy-ocean-wit...,en-us,custom,2025-10-17T09:00:00Z,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [104]:
# confirming column names
print(df_classifications.columns.tolist())
print(df_venues.columns.tolist())
pprint(df_attractions.columns.tolist())

['id', 'segment_id', 'segment_name', 'genre_id', 'genre_name', 'subGenre_id', 'subGenre_name', 'family', 'primary', 'type_id', 'type_name', 'subType_id', 'subType_name']
['id', 'name', 'type', 'test', 'url', 'locale', 'postalCode', 'timezone', 'city_name', 'country_name', 'country_countryCode', 'address_line1', 'location_longitude', 'location_latitude', 'upcomingEvents_universe', 'upcomingEvents__total', 'upcomingEvents__filtered', '_links_self', 'address_line2', 'upcomingEvents_ticketmaster', 'ada_adaPhones', 'ada_adaCustomCopy', 'ada_adaHours', 'boxOfficeInfo_openHoursDetail', 'boxOfficeInfo_willCallDetail', 'parkingDetail', 'accessibleSeatingDetail', 'generalInfo_generalRule', 'generalInfo_childRule', 'boxOfficeInfo_acceptedPaymentDetail', 'boxOfficeInfo_phoneNumberDetail', 'social_twitter', 'upcomingEvents_tmr', 'upcomingEvents_tmc', 'upcomingEvents_sportxr-uk_asmglobalmanchester']
['id',
 'name',
 'type',
 'test',
 'url',
 'locale',
 'externalLinks_spotify',
 'externalLinks_instag

Creating our genre - df.genre

In [124]:
# Start with core scalar event fields from the now-clean df
df_genre = df[['id', 'name', 'dates.start.localDate', 'dates.spanMultipleDays']].copy()

# Pull genre from df_classifications
df_genre = df_genre.merge(
    df_classifications[['id', 'genre_name', 'subGenre_name']],
    on='id', how='left'
)

# Pull venue city/region from df_venues
df_genre = df_genre.merge(
    df_venues[['id', 'city_name', 'state_name']],   # adjust names once confirmed
    on='id', how='left'
)

# Pull lineup genre summary from df_attractions
attraction_genre_summary = df_attractions.groupby('id').agg(
    attraction_count=('id', 'count'),
    attraction_genres=('genre_name', lambda x: list(x.dropna().unique()))  # adjust field name once confirmed
).reset_index().rename(columns={'id': 'event_id'})

df_genre = df_genre.merge(
    attraction_genre_summary, left_on='id', right_on='event_id', how='left'
)
df_genre = df_genre.drop(columns=['event_id'])

df_genre.head()

KeyError: "['state_name'] not in index"

In [119]:
df_attractions.head()

,id,name,type,test,url,locale,externalLinks_spotify,externalLinks_instagram,upcomingEvents__total,upcomingEvents__filtered,...,upcomingEvents_mfx-pl,upcomingEvents_mfx-ae,upcomingEvents_crowder,upcomingEvents_ticketnet,upcomingEvents_tmc,upcomingEvents_tixcraft-sg,upcomingEvents_sportxr,upcomingEvents_quicket,upcomingEvents_mfx-za,upcomingEvents_sportxtr
0,K8vZ917O7if,Dewin,attraction,False,https://www.ticketmaster.com/dewin-tickets/art...,en-us,[{'url': 'https://open.spotify.com/artist/1ers...,[{'url': 'https://www.instagram.com/dewinband/'}],0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,K8vZ917L9i0,Everywhere at Once,attraction,False,https://www.ticketmaster.com/everywhere-at-onc...,en-us,NaN,NaN,2,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,K8vZ917O7if,Dewin,attraction,False,https://www.ticketmaster.com/dewin-tickets/art...,en-us,[{'url': 'https://open.spotify.com/artist/1ers...,[{'url': 'https://www.instagram.com/dewinband/'}],0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,K8vZ917L9i0,Everywhere at Once,attraction,False,https://www.ticketmaster.com/everywhere-at-onc...,en-us,NaN,NaN,2,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,K8vZ917Khtf,Llangollen International Eisteddfod,attraction,False,https://www.ticketmaster.com/llangollen-intern...,en-us,NaN,NaN,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [128]:
df_og['_embedded.attractions'].iloc[0]

[{'name': 'Dewin',
  'type': 'attraction',
  'id': 'K8vZ917O7if',
  'test': False,
  'url': 'https://www.ticketmaster.com/dewin-tickets/artist/4412873',
  'locale': 'en-us',
  'externalLinks': {'spotify': [{'url': 'https://open.spotify.com/artist/1ersOZ90l0IgAQMqSDnDvu'}],
   'instagram': [{'url': 'https://www.instagram.com/dewinband/'}]},
  'images': [{'ratio': '16_9',
    'url': 'https://s1.ticketm.net/dam/c/f50/96fa13be-e395-429b-8558-a51bb9054f50_105951_TABLET_LANDSCAPE_LARGE_16_9.jpg',
    'width': 2048,
    'height': 1152,
    'fallback': True},
   {'ratio': '3_2',
    'url': 'https://s1.ticketm.net/dam/c/797/5e693c26-2881-4776-8f0c-3aa94bfa3797_106511_TABLET_LANDSCAPE_3_2.jpg',
    'width': 1024,
    'height': 683,
    'fallback': True},
   {'ratio': '3_2',
    'url': 'https://s1.ticketm.net/dam/c/797/5e693c26-2881-4776-8f0c-3aa94bfa3797_106511_ARTIST_PAGE_3_2.jpg',
    'width': 305,
    'height': 203,
    'fallback': True},
   {'ratio': '16_9',
    'url': 'https://s1.ticketm.ne

In [109]:
sample = df_og['_embedded.attractions'].dropna().iloc[0][0]

for key in ['classifications', 'aliases', '_links', 'images']:
    if key in sample:
        print(f"{key}: {type(sample[key]).__name__}")
    else:
        print(f"{key}: not present in this sample")

name: str
type: str
id: str
test: bool
url: str
locale: str
externalLinks: dict
images: list
classifications: list
upcomingEvents: dict
_links: dict


In [136]:
df_attractions.columns

Index(['id', 'name', 'type', 'test', 'url', 'locale', 'externalLinks_spotify',
       'externalLinks_instagram', 'upcomingEvents__total',
       'upcomingEvents__filtered', '_links_self', 'upcomingEvents_universe',
       'upcomingEvents_ticketmaster', 'externalLinks_youtube',
       'externalLinks_twitter', 'externalLinks_itunes', 'externalLinks_wiki',
       'externalLinks_facebook', 'externalLinks_musicbrainz',
       'externalLinks_homepage', 'externalLinks_lastfm',
       'upcomingEvents_mfx-fi', 'upcomingEvents_mticket',
       'externalLinks_bandcamp', 'externalLinks_tiktok',
       'externalLinks_soundcloud', 'upcomingEvents_mfx-nl',
       'upcomingEvents_mfx-de', 'upcomingEvents_mfx-cz',
       'upcomingEvents_mfx-dk', 'upcomingEvents_mfx-be',
       'upcomingEvents_mfx-ch', 'upcomingEvents_trium', 'upcomingEvents_tmr',
       'upcomingEvents_wts-tr', 'upcomingEvents_ticketweb',
       'upcomingEvents_moshtix', 'draftStatus', 'upcomingEvents_mfx-no',
       'upcomingEvents_mf

The classification dict in the df_og['_embedded.attractions'] was completely removed by the build_nested_table function. So we need to unravel another layer from df_og['_embedded.attractions'] to get the classification info. Lets call this table attraction_genre

In [140]:
attraction_classification_rows = []
for _, event_row in df_og.iterrows():
    event_id = event_row['id']
    attractions_list = event_row.get('_embedded.attractions', [])
    
    if not isinstance(attractions_list, list):
        continue
    
    for attraction in attractions_list:
        attraction_id = attraction.get('id')
        classifications = attraction.get('classifications', [{}])
        c = classifications[0] if classifications else {}
        attraction_classification_rows.append({
            'event_id': event_id,
            'attraction_id': attraction_id,
            'attraction_genre': c.get('genre', {}).get('name'),
            'attraction_subGenre': c.get('subGenre', {}).get('name'),
        })

df_attraction_genres = pd.DataFrame(attraction_classification_rows)
df_attraction_genres.head()

,event_id,attraction_id,attraction_genre,attraction_subGenre
0,LvZ18QxAj1bZeL8vGSGnc,K8vZ917O7if,Alternative,Adult Alternative Pop/Rock
1,LvZ18QxAj1bZeL8vGSGnc,K8vZ917L9i0,Other,Other
2,LvZ18QxAj1bZeL8vGSGnc,K8vZ917O7if,Alternative,Adult Alternative Pop/Rock
3,LvZ18QxAj1bZeL8vGSGnc,K8vZ917L9i0,Other,Other
4,G5vHZ_dQ2CoCm,K8vZ917Khtf,Rock,Pop


Building our df_genre table

In [165]:
df_genre = df[['id', 'name', 'dates.start.localDate']].copy()

df_genre = df_genre.merge(
    df_classifications[['id', 'genre_name', 'subGenre_name']],
    on='id', how='left'
)

df_genre = df_genre.merge(
    df_venues[['event_id', 'city_name']],
    left_on='id', right_on='event_id', how='left'
)

attraction_genre_summary = df_attraction_genres.groupby('event_id').agg(
    attraction_count=('attraction_id', 'count'),
    attraction_genres=('attraction_genre', lambda x: list(x.dropna().unique()))
).reset_index().rename(columns={'event_id': 'id'})

df_genre = df_genre.merge(attraction_genre_summary, on='id', how='left')

df_genre.head()

,id,name,dates.start.localDate,genre_name,subGenre_name,event_id,city_name,attraction_count,attraction_genres
0,LvZ18QxAj1bZeL8vGSGnc,EVERYWHERE AT ONCE: Dewin,2026-06-28,Alternative,Adult Alternative Pop/Rock,LvZ18QxAj1bZeL8vGSGnc,"Narberth, Pembrokeshire",4.0,"[Alternative, Other]"
1,LvZ18QxAj1bZeL8vGSGnc,EVERYWHERE AT ONCE: Dewin,2026-06-28,Alternative,Adult Alternative Pop/Rock,LvZ18QxAj1bZeL8vGSGnc,"Narberth, Pembrokeshire",4.0,"[Alternative, Other]"
2,LvZ18QxAj1bZeL8vGSGnc,EVERYWHERE AT ONCE: Dewin,2026-06-28,Alternative,Adult Alternative Pop/Rock,LvZ18QxAj1bZeL8vGSGnc,"Narberth, Pembrokeshire",4.0,"[Alternative, Other]"
3,LvZ18QxAj1bZeL8vGSGnc,EVERYWHERE AT ONCE: Dewin,2026-06-28,Alternative,Adult Alternative Pop/Rock,LvZ18QxAj1bZeL8vGSGnc,"Narberth, Pembrokeshire",4.0,"[Alternative, Other]"
4,LvZ18QxAj1bZeL8vGSGnc,EVERYWHERE AT ONCE: Dewin,2026-06-28,Alternative,Adult Alternative Pop/Rock,LvZ18QxAj1bZeL8vGSGnc,"Narberth, Pembrokeshire",4.0,"[Alternative, Other]"


In [166]:
df_genre.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2144 entries, 0 to 2143
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     2144 non-null   object 
 1   name                   2144 non-null   object 
 2   dates.start.localDate  2144 non-null   object 
 3   genre_name             2144 non-null   object 
 4   subGenre_name          2144 non-null   object 
 5   event_id               2144 non-null   object 
 6   city_name              2144 non-null   object 
 7   attraction_count       2064 non-null   float64
 8   attraction_genres      2064 non-null   object 
dtypes: float64(1), object(8)
memory usage: 150.9+ KB


In [167]:
null_vals(df_genre)

,null,percent
attraction_count,80,3.731
attraction_genres,80,3.731
id,0,0.000
name,0,0.000
dates.start.localDate,0,0.000
genre_name,0,0.000
subGenre_name,0,0.000
event_id,0,0.000
city_name,0,0.000


In [169]:
df_genre.head

<bound method NDFrame.head of                          id  \
0     LvZ18QxAj1bZeL8vGSGnc   
1     LvZ18QxAj1bZeL8vGSGnc   
2     LvZ18QxAj1bZeL8vGSGnc   
3     LvZ18QxAj1bZeL8vGSGnc   
4     LvZ18QxAj1bZeL8vGSGnc   
...                     ...   
2139        1kuOv0o9GAuQxki   
2140          G5dzZ_khSQFyh   
2141          G5vHZ_5UMSbtO   
2142          G5vHZ_5taClWc   
2143          G5vHZ_kIknJnr   

                                                   name dates.start.localDate  \
0                             EVERYWHERE AT ONCE: Dewin            2026-06-28   
1                             EVERYWHERE AT ONCE: Dewin            2026-06-28   
2                             EVERYWHERE AT ONCE: Dewin            2026-06-28   
3                             EVERYWHERE AT ONCE: Dewin            2026-06-28   
4                             EVERYWHERE AT ONCE: Dewin            2026-06-28   
...                                                 ...                   ...   
2139                          

In [170]:
df_genre.to_csv("df_genre.csv", index=False)

In [183]:
df_og['dates.spanMultipleDays'].unique()

array([False])

In [177]:
df_og.columns.tolist()

['name',
 'type',
 'id',
 'test',
 'description',
 'url',
 'locale',
 'images',
 'classifications',
 'nameOrigin',
 'sales.public.startDateTime',
 'sales.public.startTBD',
 'sales.public.startTBA',
 'sales.public.endDateTime',
 'dates.access.startDateTime',
 'dates.access.startApproximate',
 'dates.access.endDateTime',
 'dates.access.endApproximate',
 'dates.start.localDate',
 'dates.start.localTime',
 'dates.start.dateTime',
 'dates.start.dateTBD',
 'dates.start.dateTBA',
 'dates.start.timeTBA',
 'dates.start.noSpecificTime',
 'dates.end.localTime',
 'dates.end.dateTime',
 'dates.end.approximate',
 'dates.end.noSpecificTime',
 'dates.timezone',
 'dates.status.code',
 'dates.spanMultipleDays',
 '_links.self.href',
 '_links.attractions',
 '_links.venues',
 '_embedded.venues',
 '_embedded.attractions',
 'promoters',
 'pleaseNote',
 'promoter.id',
 'promoter.name',
 'promoter.description',
 'accessibility.ticketLimit',
 'ageRestrictions.legalAgeEnforced',
 'ticketing.safeTix.enabled',
 't